# Lesson 1.3: Structured output (JSON schema, Pydantic, validate and retry)
*Build an AI Agent from Scratch with LangGraph · Chapter 1*

**Goal:** turn a free-text reply into data your code can trust: a Pydantic contract, the JSON schema it makes,
a validate-and-retry loop, and strict JSON schema mode on the server.

**What you will be able to do**
- write a Pydantic model with `Literal` and `max_length` rules and read its validation errors
- explain why `json.loads` working does not mean the reply is correct
- write a retry loop that sends the error back to the model, with a cap on attempts
- use strict JSON schema mode (and fix its `additionalProperties` error), and still validate in your code

## How to use this notebook
- Run the cells **from top to bottom** (Shift + Enter).
- Cells that start with `# calls the API` send real requests to Groq (three cells, four requests).
  Without a key they print `skipped` and the notebook uses the replies from the video.
- Each exercise has a check cell that prints `✓ Exercise N passed`; no check cell calls the API. Solutions at the end.
- **Colab:** `pip install openai pydantic python-dotenv`, and set the key with
  `from getpass import getpass; os.environ["GROQ_API_KEY"] = getpass("Groq key: ")`.

In [1]:
# Setup: run this cell first.
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))          # finds a .env in this folder or any folder above it
OFFLINE = os.environ.get("NOTEBOOK_OFFLINE") == "1"   # set NOTEBOOK_OFFLINE=1 to skip every API cell
HAVE_KEY = bool(os.environ.get("GROQ_API_KEY")) and not OFFLINE
MODEL = "openai/gpt-oss-20b"
if HAVE_KEY:
    print("GROQ_API_KEY found: cells marked 'calls the API' will really call Groq.")
else:
    print("No GROQ_API_KEY: cells marked 'calls the API' print 'skipped' and the video's numbers are used.")
    print("Get the free key in lesson 0.2, put the line GROQ_API_KEY=... in a .env file, restart the kernel.")

GROQ_API_KEY found: cells marked 'calls the API' will really call Groq.


In [2]:
# Make the course code importable: find the repo's nightwatch/ folder above this notebook.
import sys
from pathlib import Path

here = Path.cwd()
NIGHTWATCH = next((p / "nightwatch" for p in [here, *here.parents]
                   if (p / "nightwatch" / "agent_v0.py").exists()), None)
if NIGHTWATCH:
    sys.path.insert(0, str(NIGHTWATCH))
    print("nightwatch/ found at", Path(os.path.relpath(NIGHTWATCH, here)).as_posix())
else:
    print("nightwatch/ not found: open this notebook from lessons/<lesson>/notebook/ inside the course repo.")

nightwatch/ found at ../../../nightwatch


In [3]:
import json
from typing import Literal

from openai import BadRequestError, OpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

client = OpenAI(api_key=os.environ.get("GROQ_API_KEY", "none"),
                base_url="https://api.groq.com/openai/v1", max_retries=8)
ALERT = ("Alert: checkout error rate is 23%. Logs: 'upstream payments returned 500 for POST /payments/v2/charge'. "
         "Deploy at 02:12: payments v1.3.5.")

## 1. Recall (lesson 1.2)
The same question, eight times at temperature 0, gave **two** different answers. If the words can drift, the shape of
the answer can drift too, so we check every reply, every time.

In [4]:
from collections import Counter
print(Counter([102, 102, 113, 102, 102, 113, 113, 102]), "-> 2 different answers out of 8")

Counter({102: 5, 113: 3}) -> 2 different answers out of 8


## 2. Key words

| Word | Meaning |
|---|---|
| JSON | text that holds dicts and lists (lesson 1.1) |
| schema | the shape data must have: which keys, which types, which values |
| Pydantic model | a Python class that holds a schema and checks data against it |
| validation error | the report when data breaks a rule: which field, which rule |
| strict mode | the server itself forces the reply to match your schema |
| retry | asking again, this time with the error attached |

## 3. The idea and a hand trace
The reply is a **contract** between the model and your code, enforced in three layers:

| Layer | Where | What it does | What gets through |
|---|---|---|---|
| 1. ask | the prompt | says what we want | the model can ignore it |
| 2. constrain | the server (JSON schema mode) | produces the shape we describe | rules the server does not enforce |
| 3. check | our code (Pydantic) | validates every reply before we use it | nothing: fully in our hands |

Four replies through the `Triage` contract (service: str, severity: one of critical/high/low, summary ≤ 120 chars),
by hand before we run it:

| Reply | What is wrong | Error type | Where |
|---|---|---|---|
| good | nothing | (passes) | |
| `"severity": "SEV1"` | not one of the three words | `literal_error` | `('severity',)` |
| no severity key | field missing | `missing` | `('severity',)` |
| wrapped in a markdown fence | not JSON at all | `json_invalid` | `()`, line 1 column 1 |

## 4. Write it yourself
### 4.1 The hook: it parses, is it right?
We ask nicely for JSON with three keys.

In [5]:
# The reply from the video (real run), used when there is no key.
VIDEO_REPLY = ('{"service":"payments","severity":"critical","summary":"Checkout error rate spiked to 23% due to upstream '
               'payments service returning 500 errors on POST /payments/v2/charge after deployment of payments v1.3.5 '
               'at 02:12. Immediate investigation required."}')

In [6]:
# calls the API (1 request)
ask_json = [{"role": "system", "content": "Reply with JSON with keys service, severity, summary."},
            {"role": "user", "content": ALERT}]
if HAVE_KEY:
    text = client.chat.completions.create(model=MODEL, messages=ask_json, temperature=0).choices[0].message.content
else:
    print("skipped (no key): using the video's reply")
    text = VIDEO_REPLY
print("raw reply:", text[:300])
try:
    data = json.loads(text)
    print("json.loads: ok ->", list(data))
except json.JSONDecodeError as e:
    print("json.loads failed:", e)

raw reply: {"service":"payments","severity":"critical","summary":"Checkout error rate spiked to 23% due to upstream payments service returning 500 errors on POST /payments/v2/charge after deployment of payments v1.3.5 at 02:12. Immediate investigation required."}
json.loads: ok -> ['service', 'severity', 'summary']


### 4.2 The contract as a Pydantic model

In [7]:
class Triage(BaseModel):
    service: str = Field(description="the service that is most likely broken")
    severity: Literal["critical", "high", "low"]
    summary: str = Field(max_length=120)

In [8]:
def first_error(text, model=Triage):
    """Validate; return 'OK' or the first error as 'type: loc msg'."""
    try:
        model.model_validate_json(text)
        return "OK"
    except ValidationError as e:
        err = e.errors()[0]
        return f"{err['type']}: {err['loc']} {err['msg'][:70]}"

for name, s in [("video reply", VIDEO_REPLY), ("your reply", text)]:
    print(f"{name:12} summary {len(json.loads(s)['summary']) if s.lstrip().startswith('{') else '?'} chars ->", first_error(s))

video reply  summary 195 chars -> string_too_long: ('summary',) String should have at most 120 characters
your reply   summary 195 chars -> string_too_long: ('summary',) String should have at most 120 characters


Video: `json.loads` said OK, but the summary had **195** characters against a limit of 120:
`string_too_long: ('summary',)`. It parses, but that does not make it correct.

### 4.3 Four replies through the validator (no API)
Compare with the hand-trace table.

In [9]:
samples = {
    "good": '{"service": "payments", "severity": "high", "summary": "payments v1.3.5 breaks checkout"}',
    "wrong severity": '{"service": "payments", "severity": "SEV1", "summary": "payments is down"}',
    "missing key": '{"service": "payments", "summary": "payments is down"}',
    "markdown fence": '```json\n{"service": "payments", "severity": "high", "summary": "x"}\n```',
}
for name, s in samples.items():
    try:
        t = Triage.model_validate_json(s)
        print(f"{name:15} -> OK {t}")
    except ValidationError as e:
        err = e.errors()[0]
        print(f"{name:15} -> {err['type']}: {err['loc']} {err['msg'][:70]}")

good            -> OK service='payments' severity='high' summary='payments v1.3.5 breaks checkout'
wrong severity  -> literal_error: ('severity',) Input should be 'critical', 'high' or 'low'
missing key     -> missing: ('severity',) Field required
markdown fence  -> json_invalid: () Invalid JSON: expected value at line 1 column 1


### 4.4 Validate and retry, from scratch
Plan: call, validate; if it passes, return it; if not, append the bad reply (assistant) and the error (user) and call
again; at most three attempts. Attempt 1 uses a planted bad reply on purpose, so we can watch the repair.
`ask` is passed in, so the same loop can run with a fake model in the exercises.

In [10]:
SYSTEM = "Reply with JSON only: service, severity (critical|high|low), summary."
BAD = '{"service": "payments", "severity": "SEV1", "summary": "payments is down"}'

def ask(messages):
    return client.chat.completions.create(model=MODEL, messages=messages, temperature=0).choices[0].message.content

def triage(alert, attempts=3, ask=ask, plant_bad=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": alert}]
    for i in range(1, attempts + 1):
        reply = BAD if (plant_bad and i == 1) else ask(messages)
        try:
            t = Triage.model_validate_json(reply)
            print(f"attempt {i}: valid -> {t}")
            return t, i
        except ValidationError as e:
            err = e.errors()[0]
            msg = f"Your JSON was invalid: {err['loc']} {err['msg']}. Reply again with JSON only."
            print(f"attempt {i}: invalid -> {err['msg'][:70]}")
            messages += [{"role": "assistant", "content": reply}, {"role": "user", "content": msg}]
            print(f"           history now {len(messages)} messages")
    raise RuntimeError("no valid answer")

In [11]:
# calls the API (1 request: attempt 1 is planted)
VIDEO_FIX = ('{"service": "payments", "severity": "critical", "summary": "Payments service experiencing high error '
             'rate (23%) due to upstream 500 errors on /payments/v2/charge"}')
if not HAVE_KEY:
    print("skipped (no key): a stand-in model returns the video's attempt-2 reply")
result, tries = triage(ALERT, ask=ask if HAVE_KEY else (lambda m: VIDEO_FIX))
print("calls needed:", tries, "(attempt 1 was planted, so the model was called", tries - 1, "time)")

attempt 1: invalid -> Input should be 'critical', 'high' or 'low'
           history now 4 messages


attempt 2: valid -> service='payments' severity='critical' summary='Payments service experiencing high error rate (23%) due to upstream 500 errors on /payments/v2/charge'
calls needed: 2 (attempt 1 was planted, so the model was called 1 time)


Video: attempt 1 invalid (`Input should be 'critical', 'high' or 'low'`), attempt 2 sent all four messages and
came back `payments`, `critical`. Calls needed: 2.

## 5. Library version + check: the schema, and strict mode
`model_json_schema()` turns the class into a JSON schema: `Literal` becomes `enum`, `max_length` becomes `maxLength`,
and every field without a default is `required`.

In [12]:
import pydantic
print("pydantic", pydantic.VERSION)
schema = Triage.model_json_schema()
for k, v in schema["properties"].items():
    print(f"{k:9}", v)
print("required:", schema["required"])
assert schema["properties"]["severity"]["enum"] == ["critical", "high", "low"]
assert schema["properties"]["summary"]["maxLength"] == 120
print("check: enum and maxLength are in the schema")

pydantic 2.14.3
service   {'description': 'the service that is most likely broken', 'title': 'Service', 'type': 'string'}
severity  {'enum': ['critical', 'high', 'low'], 'title': 'Severity', 'type': 'string'}
summary   {'maxLength': 120, 'title': 'Summary', 'type': 'string'}
required: ['service', 'severity', 'summary']
check: enum and maxLength are in the schema


Now hand that schema to the server with `response_format`, `strict: True`. Try 1 sends the plain schema.

In [13]:
class StrictTriage(Triage):
    model_config = ConfigDict(extra="forbid")      # -> "additionalProperties": false

schema2 = StrictTriage.model_json_schema()
print("plain schema additionalProperties:", schema.get("additionalProperties"))
print("strict schema additionalProperties:", schema2.get("additionalProperties"))

def strict_call(schema):
    return client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": "Triage the alert."}, {"role": "user", "content": ALERT}],
        response_format={"type": "json_schema", "json_schema": {"name": "triage", "strict": True, "schema": schema}})

plain schema additionalProperties: None
strict schema additionalProperties: False


In [14]:
# calls the API (2 requests)
if HAVE_KEY:
    try:
        strict_call(schema)
    except BadRequestError as e:
        print("try 1, plain schema -> 400:", str(e).split(" - ", 1)[-1][:150])
    raw = strict_call(schema2).choices[0].message.content
    print("try 2 raw:", raw)
    print("validated:", StrictTriage.model_validate_json(raw))
else:
    print("skipped (no key). Video: try 1 -> 400 `additionalProperties:false` must be set on every object;")
    print("try 2 -> {\"service\":\"payments\",\"severity\":\"high\",...}, validated by Pydantic")

try 1, plain schema -> 400:

 invalid JSON schema for response_format: 'triage': `additionalProperties:false` must be set on every object


try 2 raw: {"service":"payments","severity":"high","summary":"Checkout error rate 23% due to upstream payments 500 on POST /payments/v2/charge"}
validated: service='payments' severity='high' summary='Checkout error rate 23% due to upstream payments 500 on POST /payments/v2/charge'


Keep both layers: strict mode works on the server (valid JSON, only our keys, the right types: fewer retries);
Pydantic works in our code, checks every rule every time (servers differ in which schema rules they enforce), and hands
back a typed object, so the next line can trust `triage.severity`.

## 6. Where it shows up in agents: validate first, then act
Tool arguments are JSON that must match a schema (chapter 2), and Nightwatch ends every run by calling `finish`,
whose arguments follow a schema just like `Triage`. Below: the `finish` schema from the course agent, as a Pydantic
model, checked against the `finish` arguments of all 180 recorded baseline runs (**no API**).

In [15]:
if NIGHTWATCH:
    import agent_v0
    fin = next(t["function"] for t in agent_v0.TOOLS if t["function"]["name"] == "finish")
    causes = fin["parameters"]["properties"]["cause"]["enum"]
    print("finish needs:", fin["parameters"]["required"])
    print("cause enum:", causes)

    Finish = type("Finish", (BaseModel,), {"__annotations__": {"root_cause_service": str,
                                                                "cause": Literal[tuple(causes)], "summary": str}})
    rows = [json.loads(l) for l in (NIGHTWATCH / "runs" / "baseline_gpt-oss-20b.jsonl").read_text(encoding="utf-8").splitlines()]
    verdicts = Counter()
    for r in rows:
        if r["final"] is None:
            verdicts["no finish (step limit)"] += 1
            continue
        try:
            Finish.model_validate(r["final"])
            verdicts["valid"] += 1
        except ValidationError as e:
            verdicts[e.errors()[0]["type"]] += 1
    print(dict(verdicts))
else:
    print("nightwatch/ not found, skipped")

finish needs: ['root_cause_service', 'cause', 'summary']
cause enum: ['bad_deploy', 'memory_leak', 'traffic_spike', 'bad_flag', 'stuck_consumer', 'expired_secret', 'bad_config', 'disk_full', 'false_alarm', 'unknown']
{'valid': 134, 'no finish (step limit)': 46}


Only validated arguments may drive an action: a wrong service name means rolling back the wrong thing.

## 7. Traps
**Trap 1: `json.loads` works, so the reply must be valid.** It only proves the text parses.

In [16]:
try:
    json.loads(VIDEO_REPLY)
    print("json.loads: fine")
    Triage.model_validate_json(VIDEO_REPLY)
except ValidationError as e:
    print("ValidationError:", e.errors()[0]["type"], e.errors()[0]["loc"])

json.loads: fine
ValidationError: string_too_long ('summary',)


**Trap 2: retrying with the same messages.** The model never learns what was wrong. A stand-in model that only
fixes its reply when it sees the error message:

In [17]:
def picky_model(messages):
    saw_error = any("invalid" in m["content"] for m in messages if m["role"] == "user")
    return VIDEO_FIX if saw_error else BAD

def triage_no_feedback(alert, attempts=3):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": alert}]
    for i in range(1, attempts + 1):
        try:
            return Triage.model_validate_json(picky_model(messages)), i
        except ValidationError:
            pass                                  # trap: nothing appended, same messages next time
    raise RuntimeError(f"no valid answer after {attempts} attempts")

try:
    triage_no_feedback(ALERT)
except RuntimeError as e:
    print("RuntimeError:", e)
print("with feedback:", triage(ALERT, ask=picky_model)[1], "attempts")

RuntimeError: no valid answer after 3 attempts
attempt 1: invalid -> Input should be 'critical', 'high' or 'low'
           history now 4 messages
attempt 2: valid -> service='payments' severity='critical' summary='Payments service experiencing high error rate (23%) due to upstream 500 errors on /payments/v2/charge'
with feedback: 2 attempts


**Trap 3: retrying forever.** A stubborn model burns the budget. The cap turns it into a clear error.

In [18]:
calls = []
def stubborn(messages):
    calls.append(1)
    return BAD
try:
    triage(ALERT, ask=stubborn, plant_bad=False)
except RuntimeError as e:
    print("RuntimeError:", e, "| model calls:", len(calls))

attempt 1: invalid -> Input should be 'critical', 'high' or 'low'
           history now 4 messages
attempt 2: invalid -> Input should be 'critical', 'high' or 'low'
           history now 6 messages
attempt 3: invalid -> Input should be 'critical', 'high' or 'low'
           history now 8 messages
RuntimeError: no valid answer | model calls: 3


**Trap 4: trusting strict mode for your business rules.** The server promises the *shape*; servers differ in
which rules (like `maxLength`) they enforce. A shape-only check lets the 195-character summary through; Pydantic does not.

In [19]:
class ShapeOnly(BaseModel):
    service: str
    severity: str
    summary: str

print("shape-only:", first_error(VIDEO_REPLY, ShapeOnly))
print("Triage:    ", first_error(VIDEO_REPLY, Triage))

shape-only: OK
Triage:     string_too_long: ('summary',) String should have at most 120 characters


**Trap 5: a markdown fence around the JSON.** Parsing breaks at line 1, column 1.

In [20]:
try:
    json.loads(samples["markdown fence"])
except json.JSONDecodeError as e:
    print("JSONDecodeError:", e)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)


## 8. Exercises

In [21]:
def check(n, test):
    """Run the asserts in test(). Print one clear line instead of stopping the notebook."""
    try:
        test()
    except AssertionError as e:
        print(f"✗ Exercise {n} failed: {e}")
    except Exception as e:
        print(f"✗ Exercise {n} failed: {type(e).__name__}: {e}")
    else:
        print(f"✓ Exercise {n} passed")

### Exercise 1 (from the video): add a confidence field
Make `Triage2`: everything in `Triage` plus `confidence`, a number between 0 and 1 (use `Field(ge=..., le=...)`).
Then write two replies that break it in different ways, `BREAK_A` and `BREAK_B`, and print each error type.

In [22]:
class Triage2(Triage):
    # YOUR CODE HERE
    pass

BREAK_A = "..."     # YOUR reply that breaks confidence one way
BREAK_B = "..."     # YOUR reply that breaks confidence another way

for s in (BREAK_A, BREAK_B):
    print(first_error(s, Triage2))

json_invalid: () Invalid JSON: expected value at line 1 column 1
json_invalid: () Invalid JSON: expected value at line 1 column 1


In [23]:
def test_1():
    ok = '{"service": "payments", "severity": "high", "summary": "v1.3.5 breaks checkout", "confidence": 0.8}'
    assert first_error(ok, Triage2) == "OK", f"a confidence of 0.8 should pass: {first_error(ok, Triage2)}"
    edge = '{"service": "payments", "severity": "high", "summary": "s", "confidence": 1}'
    assert first_error(edge, Triage2) == "OK", "0 and 1 are allowed (ge / le)"
    errs = []
    for s in (BREAK_A, BREAK_B):
        try:
            Triage2.model_validate_json(s)
            raise AssertionError(f"this reply should fail: {s[:60]}")
        except ValidationError as e:
            err = e.errors()[0]
            assert err["loc"] == ("confidence",), f"break confidence, not {err['loc']} ({err['type']})"
            errs.append(err["type"])
    assert errs[0] != errs[1], f"both replies give {errs[0]}: break it two different ways"

check(1, test_1)

✗ Exercise 1 failed: break confidence, not () (json_invalid)


### Exercise 2: strip the fence
Write `strip_fence(text)` that removes a markdown fence (```` ```json ```` or ```` ``` ```` on the first line and
```` ``` ```` on the last) and returns the JSON inside. Text without a fence comes back unchanged (apart from spaces).

In [24]:
def strip_fence(text):
    # YOUR CODE HERE
    return text

In [25]:
def test_2():
    t = Triage.model_validate_json(strip_fence(samples["markdown fence"]))
    assert t.severity == "high", "the fenced sample should validate after strip_fence"
    assert strip_fence(samples["good"]) == samples["good"], "no fence: unchanged"
    assert json.loads(strip_fence('```\n{"a": 1}\n```')) == {"a": 1}, "a bare ``` fence too"

check(2, test_2)

✗ Exercise 2 failed: ValidationError: 1 validation error for Triage
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='```json\n{"service": "pa...", "summary": "x"}\n```', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid


### Exercise 3: the feedback message
Write `feedback(e)` that turns a `ValidationError` into the message the retry loop sends back: it must contain the
field name and the error message, e.g. `Your JSON was invalid: ('severity',) Input should be ... Reply again with JSON only.`

In [26]:
def feedback(e):
    # YOUR CODE HERE
    return ""

In [27]:
def test_3():
    try:
        Triage.model_validate_json(samples["missing key"])
    except ValidationError as e:
        m = feedback(e)
    assert "severity" in m and "Field required" in m, f"name the field and the rule: {m!r}"
    try:
        Triage.model_validate_json(samples["wrong severity"])
    except ValidationError as e:
        m = feedback(e)
    assert "'critical', 'high' or 'low'" in m, f"include Pydantic's message: {m!r}"
    assert "JSON" in m, "tell the model to reply with JSON again"

check(3, test_3)

✗ Exercise 3 failed: name the field and the rule: ''


### Exercise 4: a strict version for the server
Make `StrictTriage2` from `Triage2` so that its JSON schema has `additionalProperties: false` (what strict mode
demanded). Its schema must list all four fields as required.

In [28]:
class StrictTriage2(Triage2):
    # YOUR CODE HERE
    pass

In [29]:
def test_4():
    s = StrictTriage2.model_json_schema()
    assert s.get("additionalProperties") is False, f"additionalProperties is {s.get('additionalProperties')}"
    assert set(s["required"]) == {"service", "severity", "summary", "confidence"}, f"required: {s['required']}"
    extra = '{"service": "p", "severity": "low", "summary": "s", "confidence": 0.5, "owner": "me"}'
    assert first_error(extra, StrictTriage2).startswith("extra_forbidden"), "an extra key must now fail"

check(4, test_4)

✗ Exercise 4 failed: additionalProperties is None


## Solutions (try first!)
### Solution 1

In [30]:
class Triage2(Triage):
    confidence: float = Field(ge=0, le=1)

BREAK_A = '{"service": "payments", "severity": "high", "summary": "s", "confidence": 1.7}'
BREAK_B = '{"service": "payments", "severity": "high", "summary": "s", "confidence": "very sure"}'

for s in (BREAK_A, BREAK_B):
    print(first_error(s, Triage2))

less_than_equal: ('confidence',) Input should be less than or equal to 1
float_parsing: ('confidence',) Input should be a valid number, unable to parse string as a number


### Solution 2

In [31]:
def strip_fence(text):
    t = text.strip()
    if t.startswith("```"):
        lines = t.splitlines()
        t = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    return t.strip()

### Solution 3

In [32]:
def feedback(e):
    err = e.errors()[0]
    return f"Your JSON was invalid: {err['loc']} {err['msg']}. Reply again with JSON only."

### Solution 4

In [33]:
class StrictTriage2(Triage2):
    model_config = ConfigDict(extra="forbid")